# Module 1: Generative AI Applications
## Topics Covered
- Generative AI Fundamentals
- Prompt Engineering
- In-Context Learning
- Prompt Templates
- LangChain Framework
- Prompt Chaining
- AI Workflow Automation
- Structured AI Pipelines

---
**Real-world scenario**: Building a customer support assistant for an e-commerce platform that handles product inquiries, order status, and complaints.

## Setup & Installation

In [ ]:
# Install required packages
# Run once per environment
%pip install -q langchain langchain-openai langchain-core openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Set your OpenAI API key
# Option 1: via .env file: OPENAI_API_KEY=sk-...
# Option 2: directly here (not recommended for production)
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-api-key-here")
print("API Key loaded:", "Yes" if OPENAI_API_KEY != "your-api-key-here" else "No - please set OPENAI_API_KEY")

---
## 1. Generative AI Fundamentals

**What is Generative AI?**  
Generative AI models learn patterns from data and generate new content — text, code, images, audio. Large Language Models (LLMs) are the backbone of modern GenAI applications.

**Key concepts:**
- **Tokens**: Units of text the model processes (roughly 1 word ≈ 1.3 tokens)
- **Temperature**: Controls randomness (0 = deterministic, 1+ = creative)
- **Context Window**: Max tokens the model can process at once
- **System Prompt**: Instructions that shape model behavior

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# Initialize LLM - using gpt-4o-mini for cost efficiency
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    api_key=OPENAI_API_KEY
)

# Basic invocation - E-commerce customer support
messages = [
    SystemMessage(content="You are a helpful customer support agent for ShopEase, an e-commerce platform. Be concise and professional."),
    HumanMessage(content="My order #ORD-12345 hasn't arrived yet. It's been 7 days.")
]

response = llm.invoke(messages)
print("Customer Support Response:")
print(response.content)

---
## 2. Prompt Engineering

**Prompt Engineering** is the practice of designing effective inputs to guide LLM outputs.

### Key Techniques:
| Technique | Description |
|-----------|-------------|
| Zero-shot | No examples given, model relies on training |
| Few-shot | 2-5 examples provided in the prompt |
| Chain-of-Thought | Ask model to reason step by step |
| Role Prompting | Assign a persona/role to the model |

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

# ---- Zero-shot Prompting ----
zero_shot_msg = [
    HumanMessage(content="Classify this customer review as Positive, Negative, or Neutral:\n'The product quality is excellent but delivery was delayed by 3 days.'")
]

result = llm.invoke(zero_shot_msg)
print("Zero-shot result:", result.content)
print()

In [ ]:
# ---- Few-shot Prompting ----
few_shot_prompt = """Classify customer reviews into: Positive, Negative, Mixed.

Examples:
Review: "Fast shipping and great quality!" → Positive
Review: "Broke after one use, very disappointed." → Negative  
Review: "Good product but expensive for the size." → Mixed

Now classify:
Review: "The laptop looks amazing but the battery drains too fast."
"""

result = llm.invoke([HumanMessage(content=few_shot_prompt)])
print("Few-shot result:", result.content)

In [ ]:
# ---- Chain-of-Thought (CoT) Prompting ----
cot_prompt = """A customer ordered 3 items: a $45 shirt, a $120 jacket, and a $25 belt.
They have a 15% discount coupon and free shipping over $100.
What is the final price they pay? Think step by step."""

result = llm.invoke([HumanMessage(content=cot_prompt)])
print("Chain-of-Thought result:")
print(result.content)

---
## 3. In-Context Learning

**In-Context Learning (ICL)** allows LLMs to learn new tasks from examples provided directly in the prompt — no fine-tuning required.

**Why it matters**: You can customize LLM behavior for your domain without retraining, saving compute cost.

In [ ]:
# In-Context Learning: Teaching the model a custom response format
# Real-world use case: Standardizing customer complaint responses

icl_prompt = """You are a support agent. Always respond using this format:
EMPATHY: [Acknowledge the issue]
ACTION: [What will be done]
TIMELINE: [When it will be resolved]
OFFER: [Any compensation or goodwill gesture]

Example:
Customer: My TV stopped working after 2 weeks.
EMPATHY: I understand how frustrating it is when a new product fails so soon.
ACTION: We will send a replacement unit immediately.
TIMELINE: You will receive it within 3-5 business days.
OFFER: We're also adding a $20 store credit for the inconvenience.

Now respond:
Customer: I received the wrong item in my order and need it urgently for a gift.
"""

result = llm.invoke([HumanMessage(content=icl_prompt)])
print(result.content)

---
## 4. Prompt Templates with LangChain

**PromptTemplates** allow you to create reusable, parameterized prompts — think of them as prompt functions.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

# --- Basic PromptTemplate ---
product_review_template = PromptTemplate(
    input_variables=["product", "issue", "tone"],
    template="""Write a {tone} response to a customer complaint about {product}.
The customer reported: {issue}
Keep it under 100 words."""
)

prompt = product_review_template.format(
    product="Wireless Headphones XZ-500",
    issue="constant disconnection after 30 minutes of use",
    tone="empathetic and professional"
)
print("Generated Prompt:")
print(prompt)
print()

result = llm.invoke(prompt)
print("LLM Response:")
print(result.content)

In [ ]:
# --- ChatPromptTemplate (for chat models) ---
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role} for {company}. Always respond in {language}."),
    ("human", "{user_message}")
])

# Reusable across different departments
formatted = chat_template.format_messages(
    role="technical support specialist",
    company="TechMart",
    language="English",
    user_message="How do I reset my smart device to factory settings?"
)

result = llm.invoke(formatted)
print(result.content)

---
## 5. LangChain Framework: Prompt Chaining

**Prompt Chaining** connects multiple LLM calls where the output of one becomes the input of the next.

**Real-world use case**: Automated product description pipeline  
`Raw specs → Marketing copy → SEO optimization → Final review`

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

# Step 1: Generate marketing copy from product specs
specs_to_copy_prompt = ChatPromptTemplate.from_template(
    """Convert these product specs into engaging marketing copy (2-3 sentences):
    Product: {product_name}
    Specs: {specs}"""
)

# Step 2: Extract SEO keywords
seo_prompt = ChatPromptTemplate.from_template(
    """From this product description, extract 5 SEO keywords separated by commas:
    Description: {description}"""
)

# Step 3: Create final listing
final_listing_prompt = ChatPromptTemplate.from_template(
    """Create a complete product listing using:
    Description: {description}
    Keywords: {keywords}
    
    Format: Title, Description, Tags"""
)

# Build chains using LCEL (LangChain Expression Language)
chain1 = specs_to_copy_prompt | llm | parser
chain2 = seo_prompt | llm | parser

# Execute chain
product_name = "UltraSound Pro Earbuds"
specs = "40hr battery, ANC, IPX5 waterproof, 10mm drivers, Bluetooth 5.3, USB-C charging"

print("Step 1: Generating marketing copy...")
description = chain1.invoke({"product_name": product_name, "specs": specs})
print(description)
print()

print("Step 2: Extracting SEO keywords...")
keywords = chain2.invoke({"description": description})
print(keywords)
print()

print("Step 3: Creating final listing...")
final_chain = final_listing_prompt | llm | parser
final_listing = final_chain.invoke({"description": description, "keywords": keywords})
print(final_listing)

---
## 6. AI Workflow Automation with LCEL

**LangChain Expression Language (LCEL)** uses the pipe (`|`) operator to compose chains declaratively — enabling parallel execution, streaming, and fallbacks.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# Real-world: Parallel content generation for a product launch
# Generate social media posts for multiple platforms simultaneously

twitter_prompt = ChatPromptTemplate.from_template(
    "Write a Twitter/X post (max 280 chars) for launching: {product}"
)

linkedin_prompt = ChatPromptTemplate.from_template(
    "Write a professional LinkedIn post for launching: {product}"
)

email_prompt = ChatPromptTemplate.from_template(
    "Write a customer email announcement (2 short paragraphs) for launching: {product}"
)

# Parallel execution - all three run simultaneously
parallel_chain = RunnableParallel(
    twitter=twitter_prompt | llm | StrOutputParser(),
    linkedin=linkedin_prompt | llm | StrOutputParser(),
    email=email_prompt | llm | StrOutputParser()
)

results = parallel_chain.invoke({"product": "AI-powered Smart Home Hub v2.0"})

for platform, content in results.items():
    print(f"\n{'='*50}")
    print(f"📱 {platform.upper()} POST:")
    print('='*50)
    print(content)

---
## 7. Structured AI Pipelines with Output Parsers

**Structured outputs** ensure LLM responses conform to a specific format (JSON, Pydantic models), making them machine-processable.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List

# Define structured output schema
class ProductAnalysis(BaseModel):
    product_name: str = Field(description="Name of the product")
    sentiment: str = Field(description="Overall sentiment: positive/negative/mixed")
    pros: List[str] = Field(description="List of positive aspects")
    cons: List[str] = Field(description="List of negative aspects")
    recommended: bool = Field(description="Whether to recommend the product")
    rating_estimate: float = Field(description="Estimated rating from 1.0 to 5.0")

# Use with_structured_output (available in newer langchain-openai)
structured_llm = llm.with_structured_output(ProductAnalysis)

review_text = """
I've been using the Sony WH-1000XM5 headphones for 3 months. The noise cancellation is 
absolutely incredible and battery life exceeds the promised 30 hours. Sound quality is 
top-notch for music and calls. However, the ear pads get uncomfortable after 3+ hours, 
and the carrying case feels flimsy for the price. Also the touch controls take some 
getting used to. Overall a great product despite minor annoyances.
"""

analysis = structured_llm.invoke(f"Analyze this product review and extract structured data:\n{review_text}")

print("Structured Product Analysis:")
print(f"Product: {analysis.product_name}")
print(f"Sentiment: {analysis.sentiment}")
print(f"Rating Estimate: {analysis.rating_estimate}/5.0")
print(f"Recommended: {analysis.recommended}")
print(f"\nPros:")
for pro in analysis.pros:
    print(f"  ✅ {pro}")
print(f"\nCons:")
for con in analysis.cons:
    print(f"  ❌ {con}")

---
## Summary

| Concept | Key Takeaway |
|---------|-------------|
| Prompt Engineering | Design prompts using zero-shot, few-shot, and CoT techniques |
| In-Context Learning | Teach behavior via examples in the prompt — no fine-tuning needed |
| Prompt Templates | Parameterize prompts for reusability with `ChatPromptTemplate` |
| Prompt Chaining | Connect LLM calls to build multi-step workflows with LCEL |
| Parallel Chains | Use `RunnableParallel` to run multiple chains simultaneously |
| Structured Output | Use `with_structured_output()` + Pydantic to get structured JSON |

### Next Steps
- Module 2: RAG Applications — context-aware AI with real-time retrieval
- Explore LangChain documentation: https://python.langchain.com